# 模型可解释性：理解模型为什么这样预测
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：13_实用工具 | 源文件：模型可解释性.py | 核心功能：SHAP 值、特征重要性、局部解释

## 概述

模型可解释性回答"模型为什么做出这个预测"。SHAP 基于博弈论的 Shapley 值，是目前最可靠的解释方法。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# --- 导入库 ---
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score

## 数学原理

### 1. 特征重要性（树模型）

基于不纯度减少的总和（详见 11_特征工程/特征重要性.md）：

$$\text{Imp}(j) = \sum_{\text{nodes } t \text{ using } j} \frac{n_t}{n}\left(I(t) - \frac{n_{tL}}{n_t}I(t_L) - \frac{n_{tR}}{n_t}I(t_R)\right)$$

### 2. 排列重要性（Permutation Importance）

打乱特征 $j$ 后模型性能的下降：

$$\text{PI}(j) = s(X, y) - s(\text{perm}(X_j), y)$$

多次打乱取平均以减少随机性。

### 3. 部分依赖（Partial Dependence）

特征 $x_j$ 对预测的**边际效应**，固定其他特征为常数：

$$\text{PD}(x_j) = \frac{1}{n}\sum_{i=1}^{n} f(x_j, x_{i,-j})$$

其中 $x_{i,-j}$ 是第 $i$ 个样本除 $j$ 外的特征。PD 曲线展示 $x_j$ 与预测的平均关系。

### 4. SHAP 值

基于 Shapley 值的特征归因方法。特征 $j$ 的 SHAP 值：

$$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F|-|S|-1)!}{|F|!}\left[f(S \cup \{j\}) - f(S)\right]$$

其中 $F$ 是全部特征集，$S$ 是不包含 $j$ 的子集。

**性质**：
- **效率性**：$\sum_j \phi_j = f(x) - \mathbb{E}[f(X)]$
- **对称性**：贡献相同的特征获得相同 SHAP 值
- **可加性**：$\phi_j$ 可按特征聚合

### 5. LIME（局部可解释模型）

在样本 $x$ 附近用线性模型近似 $f$：

$$\xi(x) = \arg\min_{g \in G} L(f, g, \pi_x) + \Omega(g)$$

其中 $\pi_x(z) = \exp(-\frac{\|z-x\|^2}{\sigma^2})$ 是局部权重核，$\Omega(g)$ 是模型复杂度惩罚。

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `rf.feature_importances_` | $\hat{f}_j$（Gini 重要性） |
| `permutation_importance(model, X, y)` | $\text{PI}(j)$ |
| `PartialDependenceDisplay.from_estimator()` | $\text{PD}(x_j)$ 曲线 |
| `shap.TreeExplainer(model)` | 计算 $\phi_j$（SHAP 值） |
| `shap_values[i][j]` | 第 $i$ 个样本第 $j$ 个特征的 $\phi_j$ |

### 1. 准备数据和模型

运行 1. 准备数据和模型 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. 准备数据和模型")
print("=" * 60)

iris = load_iris()
feature_names = list(iris.feature_names)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

# 训练随机森林
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"随机森林测试准确率: {accuracy_score(y_test, rf.predict(X_test)):.4f}")

# 训练梯度提升
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
print(f"梯度提升测试准确率: {accuracy_score(y_test, gb.predict(X_test)):.4f}")

### 2. 特征重要性（树模型内置）

分析各特征对模型预测的贡献度。

In [ ]:
print("\n" + "=" * 60)
print("2. 特征重要性（树模型内置）")
print("=" * 60)

print("\n随机森林特征重要性 (基于不纯度减少):")
importances_rf = rf.feature_importances_
indices_rf = np.argsort(importances_rf)[::-1]

for i, idx in enumerate(indices_rf):
    bar_len = int(importances_rf[idx] / importances_rf.max() * 30)
    bar = '#' * bar_len
    print(f"  {feature_names[idx]:<18} {bar} {importances_rf[idx]:.4f}")

print("\n梯度提升特征重要性:")
importances_gb = gb.feature_importances_
indices_gb = np.argsort(importances_gb)[::-1]

for i, idx in enumerate(indices_gb):
    bar_len = int(importances_gb[idx] / importances_gb.max() * 30)
    bar = '#' * bar_len
    print(f"  {feature_names[idx]:<18} {bar} {importances_gb[idx]:.4f}")

### 3. 排列重要性（模型无关方法）

运行 3. 排列重要性（模型无关方法） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. 排列重要性（模型无关方法）")
print("=" * 60)

# 对随机森林计算排列重要性
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)

print("\n随机森林排列重要性:")
for i in perm_result.importances_mean.argsort()[::-1]:
    bar_len = int(perm_result.importances_mean[i] / perm_result.importances_mean.max() * 30)
    bar = '#' * bar_len
    print(f"  {feature_names[i]:<18} {bar} {perm_result.importances_mean[i]:.4f} "
# --- 继续 ---
          f"(±{perm_result.importances_std[i]:.4f})")

# 对逻辑回归也计算（说明模型无关性）
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

perm_lr = permutation_importance(lr, X_test, y_test, n_repeats=10, random_state=42)
print("\n逻辑回归排列重要性:")
for i in perm_lr.importances_mean.argsort()[::-1]:
    bar_len = int(perm_lr.importances_mean[i] / max(perm_lr.importances_mean.max(), 0.001) * 30)
    bar = '#' * bar_len
# --- 输出结果 ---
    print(f"  {feature_names[i]:<18} {bar} {perm_lr.importances_mean[i]:.4f}")

### 4. 部分依赖分析

运行 4. 部分依赖分析 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("4. 部分依赖分析")
print("=" * 60)

# 手动计算部分依赖（文本输出）
def partial_dependence_manual(model, X, feature_idx, feature_name, grid_resolution=10):
    """手动计算单个特征的部分依赖值"""
    feature_values = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), grid_resolution)
    pdp_values = []

    for val in feature_values:
        X_temp = X.copy()
        X_temp[:, feature_idx] = val
        preds = model.predict_proba(X_temp)
        pdp_values.append(preds.mean(axis=0))

    pdp_values = np.array(pdp_values)

    print(f"\n  {feature_name} 部分依赖:")
    max_bar = 25
    for i, val in enumerate(feature_values):
        # 显示每个类别的平均预测概率
        probs = pdp_values[i]
        bar = '#' * int(probs.max() * max_bar)
        print(f"    {val:>8.2f} |{bar:<{max_bar}} {probs.round(3)}")

    return feature_values, pdp_values

for feat_idx, feat_name in enumerate(feature_names):
    vals, pdp = partial_dependence_manual(rf, X_test, feat_idx, feat_name)

### 5. LIME 近似解释

运行 5. LIME 近似解释 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. LIME 近似解释（简化实现）")
print("=" * 60)

def simple_lime(model, X_train, instance, n_samples=500, feature_names=None):
    """
    简化版LIME: 在实例附近采样,用采样数据训练线性模型来近似解释
    """
    # 在实例附近生成扰动样本
    noise = np.random.randn(n_samples, X_train.shape[1]) * 0.1
    X_sample = instance + noise * np.std(X_train, axis=0)

    # 获取模型预测
    y_sample = model.predict_proba(X_sample)

    # 用线性模型拟合
    from sklearn.linear_model import Ridge
    n_classes = y_sample.shape[1]
    coefs = []

    for c in range(n_classes):
        ridge = Ridge(alpha=1.0)
        ridge.fit(X_sample, y_sample[:, c])
        coefs.append(ridge.coef_)

    coefs = np.array(coefs)

    # 输出解释
    print(f"\n  解释实例 (特征值: {instance.round(4)}):")
    if feature_names:
        # 对每个类别显示最重要的特征
        for c in range(n_classes):
            top_idx = np.argsort(np.abs(coefs[c]))[::-1][:3]
            print(f"    类别{c}:")
            for idx in top_idx:
                direction = "正向" if coefs[c][idx] > 0 else "负向"
# --- 输出结果 ---
                print(f"      {feature_names[idx]:<18} 系数={coefs[c][idx]:>+.4f} ({direction})")

    return coefs

# 解释一个测试样本
instance = X_test[0]
coefs = simple_lime(rf, X_train, instance, feature_names=feature_names)

### 6. SHAP 值解释

运行 6. SHAP 值解释 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("6. SHAP 值解释")
print("=" * 60)

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
# --- 输出结果 ---
    print("[SKIP] shap 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_SHAP = False
    print("shap 未安装, 使用简化实现")
    print("安装: pip install shap")

if HAS_SHAP:
    # 使用TreeExplainer（针对树模型优化）
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test[:5])

    print("\nSHAP值 (前5个测试样本):")
    if isinstance(shap_values, list):
        # 多分类: shap_values是列表,每个元素对应一个类别
        for sample_idx in range(min(3, len(X_test))):
            print(f"\n  样本{sample_idx+1}:")
            for c in range(len(shap_values)):
                print(f"    类别{c}:", end="")
                for f_idx, fname in enumerate(feature_names):
# --- 输出结果 ---
                    print(f" {fname}={shap_values[c][sample_idx][f_idx]:>+.4f}", end="")
                print()
    else:
        for sample_idx in range(min(3, len(X_test))):
            print(f"\n  样本{sample_idx+1}:")
# --- 循环处理 ---
            for f_idx, fname in enumerate(feature_names):
                print(f"    {fname}: {shap_values[sample_idx][f_idx]:>+.4f}")
else:
    # 简化的SHAP近似
    print("\n简化版特征贡献分析:")
    # 使用排列重要性+符号来近似SHAP
    instance = X_test[0]
    print(f"\n  样本特征值:")
    for f_idx, fname in enumerate(feature_names):
        print(f"    {fname}: {instance[f_idx]:.4f}")

    # 基于模型系数的近似贡献
    print(f"\n  预测类别: {rf.predict(instance.reshape(1, -1))[0]}")
    print(f"  各类别概率: {rf.predict_proba(instance.reshape(1, -1))[0].round(4)}")

### 7. 可解释性方法对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("7. 可解释性方法对比")
print("=" * 60)

print("""
方法              适用模型        优点                  局限
----              --------        ----                  ----
特征重要性(内置)   树模型          快速、直观            仅限树模型,对相关特征偏倚
排列重要性         任意模型        模型无关,理论扎实      计算慢,特征独立性假设
# --- 继续 ---
部分依赖           任意模型        展示特征-预测关系      只展示边际效应,忽略交互
LIME              任意模型        局部近似,直观          采样不稳定,解释可能不一致
SHAP              任意模型(树快)  博弈论基础,全局+局部   计算开销大,需要shap库

实践建议:
  1. 先用特征重要性做全局筛选
  2. 用SHAP深入分析单个预测
  3. 用部分依赖理解特征效应
  4. 不同方法结果不一致时,多角度验证
# --- 继续 ---
""")

## 关键代码解释

```python
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)
```

## 注意事项

1. SHAP 计算可能很慢（精确算法指数复杂度）
2. TreeExplainer 专为树模型优化
3. 全局解释 vs 局部解释

## 总结与延伸

以上代码展示了 **模型可解释性** 的完整流程。

**解读要点**：
- 关注工具的 **输入输出格式** 是否正确
- 观察数据加载和预处理的效率
- 检查模型保存/加载后的一致性

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **LIME**：局部线性近似
- **反事实解释**：改变什么特征会改变预测
- **注意力可视化**：Transformer 的注意力权重
